# Applio One Click Gradio

กดรันเซลล์เดียวด้านล่าง แล้วรอจนขึ้นลิงก์ Gradio

In [ ]:
# @title กดรันครั้งเดียวเพื่อเปิด Applio Gradio
# @markdown ถ้า Colab ขอสิทธิ์ Google Drive ให้กดยืนยัน 1 ครั้ง แล้วปล่อยให้รันต่อ
mount_drive = True # @param {type:"boolean"}
sync_models_with_drive = True # @param {type:"boolean"}
sharing_method = "gradio" # @param ["gradio", "localtunnel", "ngrok"]
ngrok_token = "" # @param {type:"string"}
tensorboard = False # @param {type:"boolean"}
language = "th_TH" # @param ["th_TH", "en_US"]

from IPython.display import clear_output
from pathlib import Path
import os

repo_url = "https://github.com/IAHispano/Applio/"
repo_name = "Applio"
repo_path = Path('/content') / repo_name
logs_path = repo_path / 'logs'
backups_path = Path('/content/drive/MyDrive/ApplioBackup')

if mount_drive:
    from google.colab import drive
    drive.mount('/content/drive')

%cd /content
!git config --global advice.detachedHead false
if repo_path.exists():
    %cd /content/Applio
    !git checkout -- tabs/inference/inference.py
    !git pull
else:
    !git clone {repo_url}
    %cd /content/Applio

clear_output()
print('กำลังติดตั้งของจำเป็น รอสักครู่...')
!apt update -y
!apt install -y portaudio19-dev
!curl -LsSf https://astral.sh/uv/install.sh | sh
!uv pip install -q -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match
!uv pip install -q ngrok jupyter-ui-poll
!npm install -g -q localtunnel > /dev/null 2>&1
!python core.py prerequisites --models --pretraineds-hifigan

if mount_drive and sync_models_with_drive and Path('/content/drive').is_mount():
    print('กำลังเชื่อมโฟลเดอร์โมเดลกับ Google Drive...')
    backups_path.mkdir(parents=True, exist_ok=True)
    logs_path.mkdir(parents=True, exist_ok=True)
    temp_names = {'mute', 'reference', 'zips', 'mute_spin', 'mute_spin-v2'}
    for item in list(logs_path.iterdir()):
        if item.name not in temp_names and item.is_dir() and not item.is_symlink():
            target = backups_path / item.name
            if not target.exists():
                !mv "{item}" "{target}"
    for item in list(backups_path.iterdir()):
        link = logs_path / item.name
        if item.name not in temp_names and not link.exists():
            !ln -s "{item}" "{link}"

for name in ['mute', 'reference', 'zips', 'mute_spin', 'mute_spin-v2']:
    path = logs_path / name
    if path.exists() and (path.is_symlink() or not path.is_dir()):
        !rm -rf "{path}"
    path.mkdir(parents=True, exist_ok=True)

clear_output()
print('ติดตั้งเสร็จแล้ว กำลังเปิด Gradio...')

# ตั้งภาษาหน้าเว็บ (Applio มีภาษาไทยในตัว: assets/i18n/languages/th_TH.json)
import json
os.environ['LANG'] = f'{language}.UTF-8'
config_path = repo_path / 'assets' / 'config.json'
template_path = repo_path / 'assets' / 'config_template.json'
if not config_path.exists() and template_path.exists():
    import shutil
    shutil.copy(template_path, config_path)  # รันครั้งแรกยังไม่มีไฟล์ตั้งค่า สร้างจากต้นแบบก่อน
if config_path.exists():
    config = json.loads(config_path.read_text(encoding='utf-8'))
    config.setdefault('lang', {})
    config['lang']['override'] = True
    config['lang']['selected_lang'] = language
    config_path.write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding='utf-8')

# แก้เสียงที่แปลงแล้วเล่นไม่ขึ้น/เล่นเสียงของโมเดลก่อนหน้า:
# Applio บันทึกผลด้วยชื่อเดิมทุกครั้ง เว็บและเบราว์เซอร์เลยเปิดไฟล์เก่าที่จำไว้ -> คัดลอกเป็นชื่อใหม่ทุกครั้ง (ชื่อเพลง_ชื่อโมเดล_เวลา)
inference_py = repo_path / 'tabs' / 'inference' / 'inference.py'
old_code = """                result = run_infer_script(*args)
                gr.Info(result[0])
                return result
"""
new_code = """                result = run_infer_script(*args)
                gr.Info(result[0])
                try:
                    import shutil, time, re as _re
                    src_path = result[1]
                    model = next((a for a in args if isinstance(a, str) and a.endswith('.pth')), '')
                    model = os.path.splitext(os.path.basename(model))[0]
                    stem, ext = os.path.splitext(os.path.basename(src_path))
                    stem = _re.sub(r'_output$', '', stem)
                    name = '_'.join(x for x in [stem, model, time.strftime('%H%M%S')] if x) + ext
                    new_path = os.path.join(os.path.dirname(src_path), name)
                    shutil.copy(src_path, new_path)
                    result = (result[0], new_path)
                except Exception:
                    traceback.print_exc()
                return result
"""
text = inference_py.read_text(encoding='utf-8')
if old_code in text:
    inference_py.write_text(text.replace(old_code, new_code, 1), encoding='utf-8')
    print('แก้ปัญหาเสียงเล่นไฟล์เก่าแล้ว')
else:
    print('คำเตือน: Applio เปลี่ยนโค้ด หาจุดแก้เสียงเล่นไฟล์เก่าไม่เจอ (ใช้งานต่อได้ตามปกติ)')

# ปลั๊กอิน (แท็บ Plugins): MultiModelDownload = โหลดโมเดลหลายตัวจากลิงก์ / MultiModelConvert = แปลงเพลงเดียวด้วยหลายโมเดล
plugin_sources = {'MultiModelDownload': '# ปลั๊กอิน Applio: วางลิงก์ Hugging Face แล้วเลือกโหลดโมเดลหลายตัวพร้อมกัน\nimport os\nimport re\nimport shutil\nimport requests\nimport gradio as gr\n\nnow_dir = os.getcwd()\nlogs_dir = os.path.join(now_dir, "logs")\nzips_dir = os.path.join(logs_dir, "zips")\ndrive_backup = "/content/drive/MyDrive/ApplioBackup"\n\n\ndef parse_link(link):\n    m = re.match(r"https?://huggingface\\.co/([^/\\s]+/[^/\\s]+)(?:/(?:tree|blob|resolve)/([^/\\s]+)/?(.*))?", link.strip())\n    if not m:\n        raise gr.Error("ลิงก์ไม่ถูกต้อง ต้องเป็นลิงก์ huggingface.co เช่น https://huggingface.co/ชื่อ/ชื่อโมเดล/tree/main")\n    return m.group(1), m.group(2) or "main", (m.group(3) or "").strip("/")\n\n\ndef list_files(link):\n    repo, rev, sub = parse_link(link)\n    url = f"https://huggingface.co/api/models/{repo}/tree/{rev}" + (f"/{sub}" if sub else "") + "?recursive=1"\n    r = requests.get(url, timeout=30)\n    if r.status_code != 200:\n        raise gr.Error(f"เปิดลิงก์ไม่ได้ (รหัส {r.status_code}) ตรวจว่าลิงก์ถูกและเป็นแบบสาธารณะ")\n    files = [f for f in r.json() if f.get("type") == "file" and f["path"].lower().endswith(".zip")]\n    if not files:\n        raise gr.Error("ไม่พบไฟล์ .zip ในลิงก์นี้")\n    choices = [(f"{f[\'path\']}  ({f.get(\'size\', 0) / 1024 / 1024:.0f} MB)", f["path"]) for f in files]\n    return (\n        gr.update(choices=choices, value=[], visible=True),\n        {"repo": repo, "rev": rev, "_all": [f["path"] for f in files]},\n        f"พบ {len(files)} โมเดล ติ๊กเลือกตัวที่ต้องการ แล้วกด \\"ดาวน์โหลดที่เลือก\\"",\n    )\n\n\ndef download_selected(selected, source, progress=gr.Progress()):\n    from rvc.lib.tools.model_download import handle_extraction_process\n\n    if not source:\n        raise gr.Error("กรุณากด \\"แสดงรายการโมเดล\\" ก่อน")\n    if not selected:\n        raise gr.Error("กรุณาติ๊กเลือกอย่างน้อย 1 โมเดล")\n    os.makedirs(zips_dir, exist_ok=True)\n    report = []\n    for i, path in enumerate(selected):\n        name = os.path.basename(path)\n        progress(i / len(selected), desc=f"กำลังโหลด {name} ({i + 1}/{len(selected)})")\n        try:\n            url = f"https://huggingface.co/{source[\'repo\']}/resolve/{source[\'rev\']}/{path}"\n            with requests.get(url, stream=True, timeout=60) as r:\n                r.raise_for_status()\n                with open(os.path.join(zips_dir, name), "wb") as f:\n                    for chunk in r.iter_content(1024 * 1024):\n                        f.write(chunk)\n            result = handle_extraction_process()\n            if result == "Error":\n                raise RuntimeError("แตกไฟล์ zip ไม่สำเร็จ")\n            keep_in_drive(os.path.splitext(name)[0])\n            report.append(f"✅ {name}")\n        except Exception as e:\n            report.append(f"❌ {name} : {e}")\n    progress(1, desc="เสร็จแล้ว")\n    ok = sum(r.startswith("✅") for r in report)\n    return f"โหลดสำเร็จ {ok} จาก {len(selected)} โมเดล\\n" + "\\n".join(report) + \\\n        "\\n\\nไปที่แท็บแปลงเสียง แล้วกดปุ่มรีเฟรชรายการโมเดล จะเห็นโมเดลใหม่"\n\n\ndef keep_in_drive(zip_stem):\n    # ถ้าเชื่อม Google Drive ไว้ ย้ายโมเดลไปเก็บใน Drive แล้วทำทางลัดไว้ที่เดิม (รอบหน้าไม่ต้องโหลดใหม่)\n    if not os.path.isdir(drive_backup):\n        return\n    from rvc.lib.utils import format_title\n    folder = os.path.join(logs_dir, format_title(zip_stem))\n    target = os.path.join(drive_backup, os.path.basename(folder))\n    if os.path.isdir(folder) and not os.path.islink(folder) and not os.path.exists(target):\n        shutil.move(folder, target)\n        os.symlink(target, folder)\n\n\ndef applio_plugin():\n    gr.Markdown("## โหลดโมเดลหลายตัวพร้อมกัน\\nวางลิงก์ Hugging Face (หน้าที่มีไฟล์ .zip หลายไฟล์) แล้วกด \\"แสดงรายการโมเดล\\"")\n    source = gr.State({})\n    with gr.Row():\n        link = gr.Textbox(label="ลิงก์ Hugging Face", placeholder="https://huggingface.co/ชื่อ/ชื่อโมเดล/tree/main", scale=4)\n        list_btn = gr.Button("แสดงรายการโมเดล", variant="primary", scale=1)\n    picks = gr.CheckboxGroup(label="เลือกโมเดลที่จะโหลด (เลือกได้หลายอัน)", choices=[], visible=False)\n    with gr.Row():\n        all_btn = gr.Button("เลือกทั้งหมด")\n        none_btn = gr.Button("ล้างที่เลือก")\n        dl_btn = gr.Button("ดาวน์โหลดที่เลือก", variant="primary")\n    status = gr.Textbox(label="สถานะ", lines=8, interactive=False)\n\n    list_btn.click(list_files, [link], [picks, source, status])\n    link.submit(list_files, [link], [picks, source, status])\n    all_btn.click(lambda s: gr.update(value=s.get("_all", [])), [source], [picks])\n    none_btn.click(lambda: gr.update(value=[]), None, [picks])\n    dl_btn.click(download_selected, [picks, source], [status])\n', 'MultiModelConvert': '# ปลั๊กอิน Applio: แปลงเพลงเดียวด้วยหลายโมเดล (ทำทีละโมเดล) แล้วส่งออกรวมเป็น zip\nimport os\nimport re\nimport shutil\nimport zipfile\nimport gradio as gr\n\nnow_dir = os.getcwd()\nlogs_dir = os.path.join(now_dir, "logs")\nout_root = os.path.join(now_dir, "assets", "audios", "multi_model_output")\ndrive_out = "/content/drive/MyDrive/ApplioOutput"\nMAX_PLAYERS = 12  # ตัวเล่นเสียงสูงสุดต่อรอบ (เกินนี้โหลดจาก zip)\nskip_folders = {"zips", "mute", "reference", "mute_spin", "mute_spin-v2"}\n\n\ndef clean_name(text):\n    # ชื่อไฟล์อ่านง่าย: เก็บตัวอักษร/ตัวเลข (รวมภาษาไทย) - _ ; ที่เหลือเปลี่ยนเป็น -\n    text = re.sub(r"[^\\w\\-\u0e00-\u0e7f]+", "-", text.strip())\n    return re.sub(r"-{2,}", "-", text).strip("-_") or "audio"\n\n\ndef find_models():\n    models = {}\n    for root, dirs, files in os.walk(logs_dir, followlinks=True):\n        dirs[:] = [d for d in dirs if d not in skip_folders]\n        for f in files:\n            if not f.endswith(".pth") or f.startswith(("G_", "D_")):\n                continue\n            indexes = sorted(\n                (i for i in os.listdir(root) if i.endswith(".index")),\n                key=lambda i: ("trained" in i, i),  # ใช้ไฟล์ added ก่อน trained\n            )\n            name = os.path.splitext(f)[0]\n            if name in models:\n                name = f"{os.path.basename(root)}-{name}"\n            models[name] = (os.path.join(root, f), os.path.join(root, indexes[0]) if indexes else "")\n    return dict(sorted(models.items(), key=lambda kv: kv[0].lower()))\n\n\ndef unique_path(path):\n    base, ext = os.path.splitext(path)\n    n = 2\n    while os.path.exists(path):\n        path = f"{base}({n}){ext}"\n        n += 1\n    return path\n\n\ndef refresh(search, selected):\n    # ตัวที่ติ๊กไว้แล้วยังแสดงอยู่เสมอ แม้ค้นหาคำอื่น จะได้ไม่หลุดจากที่เลือก\n    selected = [n for n in (selected or []) if n in find_models()]\n    names = [n for n in find_models() if search.lower() in n.lower()]\n    shown = selected + [n for n in names if n not in selected]\n    return gr.update(choices=shown, value=selected), f"พบ {len(names)} โมเดล · เลือกไว้ {len(selected)}"\n\n\ndef convert_all(audio, selected, pitch, f0_method, index_rate, protect, volume_envelope,\n                split_audio, clean_audio, export_format, progress=gr.Progress()):\n    from core import run_infer_script\n\n    if not audio:\n        raise gr.Error("กรุณาใส่ไฟล์เพลงก่อน")\n    if not selected:\n        raise gr.Error("กรุณาติ๊กเลือกอย่างน้อย 1 โมเดล")\n    models = find_models()\n    song = clean_name(os.path.splitext(os.path.basename(audio))[0])\n    out_dir = os.path.join(out_root, song)\n    os.makedirs(out_dir, exist_ok=True)\n    ext = export_format.lower()\n\n    results, report = [], []\n    for i, name in enumerate(selected):\n        progress(i / len(selected), desc=f"กำลังแปลงด้วย {name} ({i + 1}/{len(selected)})")\n        try:\n            pth, index = models[name]\n            temp_wav = os.path.join(out_dir, f"_temp_{i}.wav")\n            _, made = run_infer_script(\n                pitch=int(pitch), index_rate=index_rate, volume_envelope=volume_envelope,\n                protect=protect, f0_method=f0_method, input_path=audio, output_path=temp_wav,\n                pth_path=pth, index_path=index, split_audio=split_audio,\n                f0_autotune=False, f0_autotune_strength=1.0, proposed_pitch=False,\n                proposed_pitch_threshold=155.0, clean_audio=clean_audio, clean_strength=0.5,\n                export_format=export_format, embedder_model="contentvec",\n            )\n            final = unique_path(os.path.join(out_dir, f"{song}_{clean_name(name)}.{ext}"))\n            shutil.move(made, final)\n            if os.path.exists(temp_wav):\n                os.remove(temp_wav)\n            results.append(final)\n            report.append(f"✅ {os.path.basename(final)}")\n        except Exception as e:\n            report.append(f"❌ {name} : {e}")\n\n    zip_path = None\n    if results:\n        zip_path = unique_path(os.path.join(out_root, f"{song}_{len(results)}โมเดล.zip"))\n        with zipfile.ZipFile(zip_path, "w") as z:\n            for f in results:\n                z.write(f, os.path.basename(f))\n        if os.path.isdir(os.path.dirname(drive_out)):\n            os.makedirs(drive_out, exist_ok=True)\n            shutil.copy(zip_path, drive_out)\n            report.append(f"\\nเก็บสำเนาไว้ใน Google Drive: ApplioOutput/{os.path.basename(zip_path)}")\n    progress(1, desc="เสร็จแล้ว")\n    head = f"แปลงสำเร็จ {len(results)} จาก {len(selected)} โมเดล"\n    # ตัวเล่นเสียงของแต่ละโมเดล (ชื่อโมเดลกำกับ) ให้ฟังเทียบได้เลย\n    players = []\n    for f in results[:MAX_PLAYERS]:\n        label = re.sub(r"\\(\\d+\\)$", "", os.path.splitext(os.path.basename(f))[0].split("_", 1)[-1])\n        players.append(gr.update(value=f, label=f"🎧 {label}", visible=True))\n    players += [gr.update(value=None, visible=False)] * (MAX_PLAYERS - len(players))\n    if len(results) > MAX_PLAYERS:\n        head += f" (ฟังในหน้านี้ได้ {MAX_PLAYERS} ตัวแรก ที่เหลือโหลดจาก zip)"\n    return [head + "\\n" + "\\n".join(report), zip_path, results] + players\n\n\ndef applio_plugin():\n    gr.Markdown("## แปลงเพลงเดียวด้วยหลายโมเดล\\nใส่เพลง → ติ๊กโมเดล → กด \\"แปลงทั้งหมด\\" ระบบจะแปลงทีละโมเดล แล้วรวมไฟล์เป็น zip ให้โหลดทีเดียว")\n    audio = gr.Audio(label="เพลงที่จะแปลง (ควรเป็นเสียงร้องที่แยกดนตรีออกแล้ว)", type="filepath")\n    with gr.Row():\n        search = gr.Textbox(label="ค้นหาโมเดล", placeholder="เช่น RAMI", scale=3)\n        refresh_btn = gr.Button("รีเฟรชรายการโมเดล", scale=1)\n    picks = gr.CheckboxGroup(label="เลือกโมเดล (เลือกได้หลายอัน)", choices=list(find_models()))\n    found = gr.Markdown(f"พบ {len(find_models())} โมเดล")\n    with gr.Row():\n        all_btn = gr.Button("เลือกทั้งหมดที่แสดง")\n        none_btn = gr.Button("ล้างที่เลือก")\n    with gr.Accordion("ตั้งค่าการแปลง (ใช้กับทุกโมเดล)", open=True):\n        pitch = gr.Slider(-24, 24, value=0, step=1, label="ปรับคีย์ (Pitch)", info="ชายเป็นหญิง ประมาณ +12 / หญิงเป็นชาย ประมาณ -12")\n        f0_method = gr.Radio(["rmvpe", "crepe", "crepe-tiny", "fcpe", "swift"], value="rmvpe", label="วิธีจับโน้ต", info="rmvpe เหมาะกับงานส่วนใหญ่")\n        index_rate = gr.Slider(0, 1, value=0.75, step=0.01, label="ความเหมือนเสียงต้นแบบ (Index rate)")\n        protect = gr.Slider(0, 0.5, value=0.5, step=0.01, label="ป้องกันเสียงลม/พยัญชนะ (Protect)")\n        volume_envelope = gr.Slider(0, 1, value=1, step=0.01, label="ความดังตามต้นฉบับ (Volume envelope)")\n        with gr.Row():\n            split_audio = gr.Checkbox(label="แบ่งเพลงเป็นช่วงก่อนแปลง (เพลงยาว)", value=False)\n            clean_audio = gr.Checkbox(label="ลดเสียงรบกวน", value=False)\n        export_format = gr.Radio(["WAV", "MP3", "FLAC", "OGG", "M4A"], value="WAV", label="ชนิดไฟล์ผลลัพธ์")\n    go = gr.Button("แปลงทั้งหมด", variant="primary")\n    status = gr.Textbox(label="สถานะ", lines=8, interactive=False)\n    gr.Markdown("### ฟังเทียบแต่ละโมเดล")\n    players = []\n    for _ in range(MAX_PLAYERS // 2):\n        with gr.Row():\n            players.append(gr.Audio(label="", visible=False, interactive=False))\n            players.append(gr.Audio(label="", visible=False, interactive=False))\n    zip_file = gr.File(label="โหลดทุกไฟล์ (zip)")\n    files = gr.File(label="หรือโหลดทีละไฟล์", file_count="multiple")\n\n    search.change(refresh, [search, picks], [picks, found])\n    refresh_btn.click(refresh, [search, picks], [picks, found])\n    all_btn.click(lambda s, cur: gr.update(value=list(dict.fromkeys((cur or []) + [n for n in find_models() if s.lower() in n.lower()]))), [search, picks], [picks])\n    none_btn.click(lambda: gr.update(value=[]), None, [picks])\n    go.click(convert_all,\n             [audio, picks, pitch, f0_method, index_rate, protect, volume_envelope, split_audio, clean_audio, export_format],\n             [status, zip_file, files] + players)\n'}
for plugin_name, plugin_code in plugin_sources.items():
    plugin_dir = repo_path / 'tabs' / 'plugins' / 'installed' / plugin_name
    plugin_dir.mkdir(parents=True, exist_ok=True)
    (plugin_dir / '__init__.py').write_text('', encoding='utf-8')
    (plugin_dir / 'plugin.py').write_text(plugin_code, encoding='utf-8')

if tensorboard:
    %load_ext tensorboard
    %tensorboard --logdir logs --bind_all

if sharing_method == 'gradio':
    !python app.py --listen --share --client
elif sharing_method == 'localtunnel':
    !echo Password IP: $(curl --silent https://ipv4.icanhazip.com)
    !lt --port 6969 & python app.py --listen --client
elif sharing_method == 'ngrok':
    import ngrok
    ngrok.kill()
    listener = await ngrok.forward(6969, authtoken=ngrok_token)
    print(f'Ngrok URL: {listener.url()}')
    !python app.py --listen --client
